# 256×256 NFFA-SEM box inpainting

Run this after training the center-resized 256×256 model. The notebook selects the latest run automatically, uses model_final.pt when present, and falls back to the newest periodic model checkpoint. It tests one deterministic patch from the held-out test split with a centered 64×64 missing box.

The inference steps and reporting follow the 512×512 box-inpainting notebook. Change the settings below to select a run, checkpoint, test patch, or box size.


In [ ]:
from pathlib import Path
import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch

repo_root = Path.cwd().resolve()
if not (repo_root / "pnpflow").is_dir():
    candidate = repo_root.parent
    if (candidate / "pnpflow").is_dir():
        repo_root = candidate
    else:
        raise RuntimeError("Open this notebook from the PnP-Flow repository or its notebooks directory.")
os.chdir(repo_root)

DATASET_NAME = "sem_nffa_all10_centerresize_256"
MODEL_NAME = "ot"
RUN_NAME = None         # Set a run directory name to pin a specific run.
CHECKPOINT_NAME = None  # Set e.g. model_20.pt to pin a checkpoint.
TEST_SAMPLE_INDEX = 0
BOX_SIZE = 64
STEPS_PNP = 20
NUM_SAMPLES = 3
NOISE_STD = 0.01
LR_PNP = 1.0
ALPHA = 1.0
USE_AMP = True

model_root = repo_root / "model" / DATASET_NAME / MODEL_NAME
if RUN_NAME is None:
    latest_run_file = model_root / "latest_run.txt"
    if not latest_run_file.is_file():
        raise FileNotFoundError(f"No latest run file at {latest_run_file}; set RUN_NAME manually.")
    RUN_NAME = latest_run_file.read_text().strip()
run_dir = model_root / RUN_NAME

if CHECKPOINT_NAME is not None:
    checkpoint_path = run_dir / CHECKPOINT_NAME
else:
    final_checkpoint = run_dir / "model_final.pt"
    periodic_checkpoints = [
        path for path in run_dir.glob("model_*.pt")
        if path.stem.removeprefix("model_").isdigit()
    ]
    periodic_checkpoints.sort(key=lambda path: int(path.stem.removeprefix("model_")))
    if final_checkpoint.is_file():
        checkpoint_path = final_checkpoint
    elif periodic_checkpoints:
        checkpoint_path = periodic_checkpoints[-1]
        warnings.warn(f"No final checkpoint yet; using {checkpoint_path.name}.")
    else:
        raise FileNotFoundError(f"No model checkpoint found in {run_dir}")
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

test_root = repo_root / "data" / DATASET_NAME / "test"
test_files = sorted(test_root.rglob("*.npy"))
if not test_files:
    raise FileNotFoundError(f"No test patches found under {test_root}")
if not 0 <= TEST_SAMPLE_INDEX < len(test_files):
    raise IndexError(f"TEST_SAMPLE_INDEX={TEST_SAMPLE_INDEX}, but found {len(test_files)} test patches.")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
if device.type == "cuda":
    torch.cuda.manual_seed_all(0)

print(f"Run: {RUN_NAME}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Test patch: {test_files[TEST_SAMPLE_INDEX]}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(device)}")


In [ ]:
from pnpflow.utils import define_model
from pnpflow.degradations import FixedMaskInpainting
from matplotlib.patches import Rectangle

class ModelArgs:
    model = MODEL_NAME
    num_channels = 1
    dim_image = 256

model, _ = define_model(ModelArgs())
state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.to(device).eval()

patch_path = test_files[TEST_SAMPLE_INDEX]
patch = np.load(patch_path).astype(np.float32)
if patch.ndim == 3 and patch.shape[0] == 1:
    patch = patch[0]
if patch.shape != (256, 256):
    raise ValueError(f"Expected a 256 x 256 test patch, got {patch.shape} from {patch_path}")
clean = torch.from_numpy(np.clip(patch, -1.0, 1.0))[None, None].to(device=device)

if not 0 < BOX_SIZE < 256:
    raise ValueError("BOX_SIZE must be between 1 and 255.")
BOX_TOP = (256 - BOX_SIZE) // 2
BOX_LEFT = (256 - BOX_SIZE) // 2
known_mask = torch.ones_like(clean)
known_mask[:, :, BOX_TOP:BOX_TOP + BOX_SIZE, BOX_LEFT:BOX_LEFT + BOX_SIZE] = 0
missing_mask = known_mask == 0
degradation = FixedMaskInpainting(known_mask)
observed = degradation.H(clean)

clean_np = clean[0, 0].detach().cpu().numpy()
observed_np = observed[0, 0].detach().cpu().numpy()
missing_np = missing_mask[0, 0].detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
axes[0].imshow(clean_np, cmap="gray", vmin=-1, vmax=1)
axes[0].set_title("Clean held-out patch")
axes[1].imshow(observed_np, cmap="gray", vmin=-1, vmax=1)
axes[1].set_title("Input with box removed")
axes[2].imshow(missing_np, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Missing-pixel mask")
for ax in axes:
    ax.axis("off")
for ax in axes[:2]:
    ax.add_patch(Rectangle((BOX_LEFT, BOX_TOP), BOX_SIZE, BOX_SIZE, fill=False, edgecolor="red", linewidth=1.5))
plt.show()

print(f"Box: y={BOX_TOP}:{BOX_TOP + BOX_SIZE}, x={BOX_LEFT}:{BOX_LEFT + BOX_SIZE}")
print(f"Missing area: {100 * missing_mask.float().mean().item():.2f}%")


In [ ]:
@torch.inference_mode()
def pnp_flow_box_inpaint(model, observed, degradation):
    known = degradation.H_adj(torch.ones_like(observed)).to(observed.device)
    x = degradation.H_adj(observed).clone()
    lr = NOISE_STD ** 2 * LR_PNP
    delta = 1.0 / STEPS_PNP
    amp_enabled = USE_AMP and observed.device.type == "cuda"

    for iteration in range(STEPS_PNP):
        t = torch.full((len(x),), delta * iteration, device=x.device, dtype=x.dtype)
        lr_t = lr * (1.0 - t).pow(ALPHA).view(-1, 1, 1, 1)
        z = x - lr_t * degradation.H_adj(degradation.H(x) - observed) / (NOISE_STD ** 2)
        x_new = torch.zeros_like(x)

        for _ in range(NUM_SAMPLES):
            t_view = t.view(-1, 1, 1, 1)
            z_tilde = t_view * z + (1.0 - t_view) * torch.randn_like(z)
            if amp_enabled:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    velocity = model(z_tilde, t)
            else:
                velocity = model(z_tilde, t)
            x_new += (z_tilde + (1.0 - t_view) * velocity).float()

        x_new /= NUM_SAMPLES
        x = known * observed + (1.0 - known) * x_new
        if (iteration + 1) % 5 == 0 or iteration == 0:
            print(f"PnP iteration {iteration + 1}/{STEPS_PNP}")
    return x

start_time = time.perf_counter()
reconstruction = pnp_flow_box_inpaint(model, observed, degradation)
if device.type == "cuda":
    torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - start_time

reconstruction_np = reconstruction[0, 0].detach().cpu().numpy()
error = reconstruction_np - clean_np
masked_mse = float(np.mean(error[missing_np] ** 2))
zero_fill_mse = float(np.mean(clean_np[missing_np] ** 2))
observed_pixels = ~missing_np
data_consistency_error = float(
    np.max(np.abs(reconstruction_np[observed_pixels] - observed_np[observed_pixels]))
)
masked_psnr = float(10.0 * np.log10(4.0 / max(masked_mse, 1e-12)))
zero_fill_psnr = float(10.0 * np.log10(4.0 / max(zero_fill_mse, 1e-12)))

print(f"Masked-region PSNR: {masked_psnr:.2f} dB (zero-fill baseline: {zero_fill_psnr:.2f} dB)")
print(f"Max observed-pixel change: {data_consistency_error:.3g}")
print(f"Inference time: {elapsed_seconds:.1f} seconds")


In [ ]:
error_display = np.where(missing_np, np.abs(error), np.nan)
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
panels = [
    (clean_np, "Clean test patch", "gray"),
    (observed_np, "Observed input", "gray"),
    (reconstruction_np, f"Reconstruction\nmasked PSNR {masked_psnr:.2f} dB", "gray"),
    (error_display, "Absolute error in box", "magma"),
]
for ax, (image, title, cmap) in zip(axes, panels):
    if cmap == "magma":
        ax.imshow(image, cmap=cmap, vmin=0, vmax=max(float(np.nanmax(image)), 1e-6))
    else:
        ax.imshow(image, cmap=cmap, vmin=-1, vmax=1)
    ax.set_title(title)
    ax.axis("off")

output_dir = repo_root / "notebooks" / "outputs" / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)
output_stem = f"box_inpainting_{checkpoint_path.stem}_test{TEST_SAMPLE_INDEX}_{patch_path.parent.name}_{patch_path.stem}"
figure_path = output_dir / f"{output_stem}.png"
reconstruction_path = output_dir / f"{output_stem}_reconstruction.npy"
metrics_path = output_dir / f"{output_stem}_metrics.json"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

metrics = {
    "checkpoint": str(checkpoint_path),
    "test_patch": str(patch_path),
    "box": {"top": BOX_TOP, "left": BOX_LEFT, "size": BOX_SIZE},
    "steps": STEPS_PNP,
    "num_samples": NUM_SAMPLES,
    "masked_psnr_db": masked_psnr,
    "zero_fill_baseline_masked_psnr_db": zero_fill_psnr,
    "max_observed_pixel_change": data_consistency_error,
    "inference_seconds": elapsed_seconds,
}
np.save(reconstruction_path, reconstruction_np.astype(np.float32))
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")
print(f"Saved figure: {figure_path}")
print(f"Saved reconstruction: {reconstruction_path}")
print(f"Saved metrics: {metrics_path}")
